# 01 — Coleta e ingestão dos arquivos originais

Os arquivos de pedidos, itens e clientes da Olist são preservados sem alteração em um volume gerenciado do Unity Catalog. O catálogo organiza a origem bruta e os destinos Silver e Gold em esquemas separados.

A leitura inicial verifica a presença, a integridade e o esquema dos CSVs antes das transformações.


## Fonte e licença

Fonte: Olist, Brazilian E-Commerce Public Dataset by Olist, Kaggle: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce. A licença indicada na fonte é CC BY-NC-SA 4.0: https://creativecommons.org/licenses/by-nc-sa/4.0/. Os dados são usados para fins acadêmicos com atribuição, e os CSVs originais permanecem preservados no volume.

A fonte descreve pedidos de 2016 a 2018. Nos arquivos utilizados, as compras vão de 04/09/2016 a 17/10/2018. Pedidos fornecem status e datas de compra, previsão e entrega; clientes fornecem localidade; itens fornecem preço e frete por item.


## Configuração do catálogo e do volume


In [ ]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalogo", "")
catalogo = dbutils.widgets.get("catalogo").strip() or spark.sql("SELECT current_catalog()").first()[0]
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo), "Informe um catálogo válido no widget."
assert catalogo != "hive_metastore", "Use o catálogo do workspace com Unity Catalog."
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
origem = f"/Volumes/{catalogo}/olist_mvp_raw/originais"
silver = f"{catalogo}.olist_mvp_silver"
gold = f"{catalogo}.olist_mvp_gold"
arquivos = {
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "clientes": "olist_customers_dataset.csv",
}
colunas_esperadas = {
    "pedidos": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "itens": ["order_id", "order_item_id", "product_id", "seller_id",
              "shipping_limit_date", "price", "freight_value"],
    "clientes": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                 "customer_city", "customer_state"],
}
print(f"Catálogo: {catalogo}; volume: {origem}")


In [ ]:
for esquema in ["olist_mvp_raw", "olist_mvp_silver", "olist_mvp_gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.olist_mvp_raw.originais")
print(f"Volume pronto: {origem}")


## Carga dos arquivos no volume

Os três CSVs originais foram enviados pela interface do Databricks ao volume gerenciado. Esse caminho evita depender de download externo no compute da Free Edition. A carga mantém os arquivos intactos; a listagem, os tamanhos, os hashes e a leitura a seguir verificam a cópia recebida.


In [ ]:
inventario = {item.name: item.size for item in dbutils.fs.ls(origem)}
faltantes = sorted(set(arquivos.values()) - set(inventario))
assert not faltantes, f"Arquivos ausentes no volume: {faltantes}"
display(spark.createDataFrame(
    [(nome, inventario[nome]) for nome in arquivos.values()],
    ["arquivo", "tamanho_bytes"]
))


In [ ]:
import hashlib

hashes_esperados = {
    "olist_orders_dataset.csv": "8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf",
    "olist_order_items_dataset.csv": "0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279",
    "olist_customers_dataset.csv": "983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2",
}
def sha256_arquivo(caminho):
    digestao = hashlib.sha256()
    with open(caminho, "rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            digestao.update(bloco)
    return digestao.hexdigest()

for nome, esperado in hashes_esperados.items():
    observado = sha256_arquivo(f"{origem}/{nome}")
    assert observado == esperado, f"Arquivo divergente: {nome}"
    print(f"{nome}: SHA-256 conferido ({observado})")


## Leitura sem conversão de tipos


In [ ]:
def ler_origem(nome):
    caminho = f"{origem}/{arquivos[nome]}"
    quadro = (spark.read.option("header", "true")
              .option("inferSchema", "false")
              .option("sep", ",")
              .option("encoding", "UTF-8")
              .option("mode", "FAILFAST")
              .csv(caminho))
    assert quadro.columns == colunas_esperadas[nome], (
        f"Esquema inesperado em {arquivos[nome]}: {quadro.columns}"
    )
    return quadro


In [ ]:
for nome in arquivos:
    quadro = ler_origem(nome)
    print(f"{nome}: {quadro.count()} linhas; colunas: {quadro.columns}")
    quadro.printSchema()
    display(quadro.limit(5))
